# Fairy2iOS — Fairy Tail 2 (Switch → iOS 26)

**Title ID:** `0100B9801F872000` · **branch:** `Fairy2iOS` · **Drive:** `fairy2`.

**Ordem:** 1) autenticar Drive; 2) clonar branch; 3) baixar e extrair os três RARs, enviar **as 18 NSPs** ao Drive, apagar RARs; 4) recuperar checkpoint se existir; 5) preparar o jogo em Linux; 6) conferir; 7) enviar handoff ao GitHub; 8) após sucesso do Actions, transferir IPA ao Drive.

O Colab realiza todos os trabalhos pesados que podem ser feitos em Linux. O Actions usa **xcode-27** apenas para host Apple Silicon, Metal, framework e assinatura ad-hoc. A IPA contém os shaders compilados e usa `romfs.img` externa. **Nenhum arquivo NSP ou chave entra no GitHub nem na IPA.**


## 1. Autenticar o Drive
Pasta principal já configurada; o jogo terá uma subpasta exclusiva `fairy2`.

In [ ]:
!python -m pip -q install google-api-python-client google-auth-httplib2 requests
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import os, sys, json, re, argparse, hashlib, subprocess, importlib, getpass

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
DRIVE_CLIENT = build("drive", "v3", credentials=credentials, cache_discovery=False)
DRIVE_PARENT_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
FAIRY2_FOLDER_ID = "1QNa5LUK7Es-evrRIAjtUVzrhGUhENn5t"
parent = DRIVE_CLIENT.files().get(fileId=DRIVE_PARENT_ID, fields="id,name,mimeType", supportsAllDrives=True).execute()
child = DRIVE_CLIENT.files().get(fileId=FAIRY2_FOLDER_ID, fields="id,name,mimeType,parents", supportsAllDrives=True).execute()
assert child["name"] == "fairy2" and DRIVE_PARENT_ID in child["parents"], "Pasta fairy2 incorreta"
print("Drive autenticado:", parent["name"], "/ fairy2")


## 2. Obter scripts atualizados da branch Fairy2iOS
Não usar os scripts `blades/`. O clone não baixa NSPs.

In [ ]:
REPO = Path("/content/Fairy2iOS-repo")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "Fairy2iOS",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "Fairy2iOS"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
COLAB_SRC = str(REPO / "fairy2" / "colab")
sys.path.insert(0, COLAB_SRC)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Branch Fairy2iOS / commit:", COMMIT)


## 3. Importar os RARs base + update + 16 DLCs para o Drive (Colab)

**Download em chunks:** 6 workers HTTP Range, 64 MiB por bloco, verificação SHA-256 por bloco e retomada de blocos já verificados. Se o link não aceitar Range, usa streaming seguro de 8 MiB. O fluxo extrai um RAR por vez, verifica NSP/PFS0, envia ao Drive em chunks resumíveis de 64 MiB (tamanho/MD5 confirmado) e apaga o RAR somente depois do upload validado. O padrão de seis workers também é usado em Drive → Colab e Drive → Actions. URLs temporárias podem expirar.


In [ ]:
# Atualiza os scripts da branch antes dos testes, preservando os arquivos do jogo.
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1",
                "origin", "Fairy2iOS"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
importlib.invalidate_caches()
for _name in ("archive_ingest", "source_archive"):
    sys.modules.pop(_name, None)
# Exibe o resultado e o traceback REAL dos testes, sem esconder stdout/stderr.
_test = subprocess.run([sys.executable, "-m", "unittest", "discover",
                        "-s", str(REPO / "fairy2" / "tests"), "-p", "test_*.py", "-v"],
                       capture_output=True, text=True, check=False)
if _test.stdout:
    print(_test.stdout, end="")
if _test.stderr:
    print(_test.stderr, end="")
if _test.returncode:
    raise RuntimeError("Pré-teste Linux falhou; detalhes acima (nenhum RAR baixado).")
import archive_ingest
BASE_RAR_URL = "https://megadl.boats/download/FAIRYTAL2-(USA)-NSwTcH-%5BBASE%5D-NSP-Ziperto.rar?download_token=Hi56pjwabuCJDyE0rm46nE2DIayFeovSOUlXgfcHq9GhtfWgy5GTRSWRbRhyCcvPr96s5uH88dx0u6klH56pjwa9smyE3XTMouKw0IJLtoyPaGm56pjwaOx56pjwaALW58HhPJF0FVG8Vb3uMkbwKsT9yhyVQZBjP6VbfKkhD56pjwaDwCAFhZqDHolQQ5UMbO0V6IQidpWKYWb56pjwaQNQvpslSfUvFCiduw4dIt6xBwrZl00DNzDwQp89hy4p89hy4"
UPDATE_RAR_URL = "https://megadl.boats/download/FAIRYTAL2-(USA)-NSwTcH-NSP-Update141-Ziperto.rar?download_token=Hi56pjwabuCJDyE0rm46nE2DIayFeovSOUlXgfcHq9GhtfWgy8A5ErB1iM4Yowu1i07y3IAF388dx0usEG6wygzzwlFTWO6dtVBMSh0CZ7x5tIPMN5qFTtjLVDoNqZdzVDUzge06GQbGfxJ4KHPlLQKny3sbopcERMoS6WPuy6CE88dx0uWasywe6LQpM8Fmpezt88dx0uCtnKvekyXUCR0cyc4yECGXXUn1yFJ6aQp89hy4p89hy4"
DLC_RAR_URL = "https://e13.megaupdownup.org/02a99380e6953d3ae7f72a8d4745d108/FAIRYTAL2-(USA)-NSwTcH-NSP-%5B16DLCPack%5D-Ziperto.rar?download_token=49aa3e19c9f57707f03b9e86f2cb8f485e02a50dbedaa4003df3477880f58a7a"
SOURCE_IMPORT = archive_ingest.ingest(
    DRIVE_CLIENT, DRIVE_PARENT_ID, BASE_RAR_URL, UPDATE_RAR_URL, DLC_RAR_URL,
    root="/content/fairy2")
print(json.dumps(SOURCE_IMPORT, indent=2, ensure_ascii=False))


## 4. Recuperar checkpoint Linux do Drive, se disponível
Evita baixar NSPs e recompilar trabalho já finalizado quando existe handoff válido de Fairy Tail 2.

In [ ]:
import run_colab
RECOVERED_HANDOFF = run_colab.recover(argparse.Namespace(drive_folder_id=DRIVE_PARENT_ID), api=DRIVE_CLIENT)
print("Checkpoint recuperado:", bool(RECOVERED_HANDOFF))


## 5. Construir perfil real a partir dos NSPs e preparar Linux
`build_ipa_from_nsp.py` verifica Title ID e versão de update, levanta perfil provisório quando não existir, faz merge RomFS, rewrite ARM64, split e decode de shaders quando suportado em Linux, cria workspace verificável, e envia RomFS externa + checkpoint ao Drive. Só operações Apple Silicon/Metal/Xcode ficam para Actions.

In [ ]:
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "Fairy2iOS"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
importlib.invalidate_caches()
run_colab = importlib.reload(run_colab)
assert run_colab.BRANCH == "Fairy2iOS" and run_colab.TITLE_ID == "0100B9801F872000"
if RECOVERED_HANDOFF:
    HANDOFF = RECOVERED_HANDOFF
    print("Reutilizando checkpoint Drive; nenhuma etapa Linux refeita.")
else:
    HANDOFF = run_colab.prepare(argparse.Namespace(drive_folder_id=DRIVE_PARENT_ID), api=DRIVE_CLIENT)
HANDOFF_PATH = Path("/content/fairy2-handoff.json")
assert HANDOFF_PATH.is_file(), "Handoff ainda não foi criado"
print("Preparação concluída:", HANDOFF["request_id"])


## 6. Gate de identidade e integridade antes do Actions
Não inventar versão do update, backend gráfico, quantidade de shaders ou perfil. Os dados devem vir do NSP real.

In [ ]:
HANDOFF = json.loads(HANDOFF_PATH.read_text())
p = HANDOFF["prepared"]
assert HANDOFF["project"] == HANDOFF["branch"] == "Fairy2iOS"
assert HANDOFF["title_id"] == p["title_id"] == "0100B9801F872000"
assert p["input_package_count"] == p["input_nsp_count"] == 18
assert p["input_update_count"] == 1 and p["input_dlc_count"] == 16
assert p["base_content_version"] == 0
assert p["update_content_version"] > 0 and HANDOFF["package_version"] == f'v{p["update_content_version"]}'
assert p["graphics"] in ("nvn","gl","vulkan")
assert HANDOFF["game_data"]["romfs"]["size"] > 0
assert len(HANDOFF["source_files"]["dlcs"]) == 16
assert len(HANDOFF["game_data"]["aoc"]) == len(p["aoc_images"])
for item in (HANDOFF["source_files"]["base"], HANDOFF["source_files"]["update"], *HANDOFF["source_files"]["dlcs"], HANDOFF["work_archive"], HANDOFF["game_data"]["romfs"], *HANDOFF["game_data"]["aoc"]):
    assert re.fullmatch("[0-9a-f]{64}", item["sha256"])
print("GATE OK; backend:", p["graphics"], "update:", HANDOFF["package_version"])
print("Perfil provisório:", p["provisional"], "shader:", p["shader_split_stage"])
print("Mac stages reais:", p["mac_host_stages"])
print("RomFS externa:", HANDOFF["game_data"]["folder_url"])


## 7. Disparar **um** workflow macOS
Publica **somente** `fairy2/handoff.json` em `Fairy2iOS`. O push aciona apenas `fairy2.yml`. Use PAT fine-grained com Contents: Read and write para KintTany; o token não é gravado no notebook.

In [ ]:
if not os.environ.get("FAIRY2_GITHUB_TOKEN"):
    os.environ["FAIRY2_GITHUB_TOKEN"] = getpass.getpass("GitHub PAT (Contents: R/W): ").strip()
assert os.environ["FAIRY2_GITHUB_TOKEN"]
run_colab.dispatch(argparse.Namespace(handoff=HANDOFF_PATH))
print("Actions: https://github.com/joao9vegeto-stack/KintTany/actions/workflows/fairy2.yml")


## 8. Transferência — somente a IPA com a correção KTGL testada no Mac
Célula independente. Atualiza a branch e importa o script por caminho absoluto, bloqueando a versão em cache. Busca apenas o último run bem-sucedido de `fairy2-ktgl-cap-repack.yml`. Confere relatório da correção testada, SHA-256 da IPA, CRC, shaders Metal, opções 30/60 FPS e inversão A/B; nunca baixa as versões antigas. Não compila.


In [ ]:
# CÉLULA 8 — INDEPENDENTE: atualiza scripts, impede cache obsoleto e exige o último host repack.
from pathlib import Path
import subprocess, os, sys, importlib.util, importlib, getpass
REPO = Path("/content/Fairy2iOS-repo")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "Fairy2iOS",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1",
                    "origin", "Fairy2iOS"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert len(COMMIT) == 40
COLAB_SRC = REPO / "fairy2" / "colab"
if str(COLAB_SRC) in sys.path:
    sys.path.remove(str(COLAB_SRC))
sys.path.insert(0, str(COLAB_SRC))
importlib.invalidate_caches()
# Carrega POR CAMINHO ABSOLUTO, não a cópia previamente importada em outro diretório.
sys.modules.pop("ipa_validator", None)
script = COLAB_SRC / "transfer_ipa.py"
source = script.read_text(encoding="utf-8")
assert "def transfer_original_explicit():" in source and "def try_transfer_ktgl_cap():" in source, (
    "Clone desatualizado: a versão antiga do Colab ainda pode selecionar o run 44965722551")
spec = importlib.util.spec_from_file_location("fairy2_latest_repack_transfer", str(script))
transfer_latest = importlib.util.module_from_spec(spec)
spec.loader.exec_module(transfer_latest)
assert Path(transfer_latest.__file__).resolve() == script.resolve()
assert transfer_latest.WORKFLOW == "fairy2.yml" and transfer_latest.BRANCH == "Fairy2iOS"
print("[Fairy2iOS] Scripts ATUALIZADOS / commit:", COMMIT, flush=True)
print("[Fairy2iOS] Transferindo SOMENTE IPA de 1,25 GiB validada no Mac; runs antigos 44965722551, 44977592488, 44980090949, 44981941022 e 44983552364 BLOQUEADOS.", flush=True)
if not os.environ.get("FAIRY2_GITHUB_TOKEN"):
    os.environ["FAIRY2_GITHUB_TOKEN"] = getpass.getpass("GitHub PAT (Actions: Read): ").strip()
assert os.environ["FAIRY2_GITHUB_TOKEN"]
# Autenticação Drive também funciona ao executar apenas a célula 8.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-api-python-client", "google-auth-httplib2", "requests"], check=True)
from google.colab import auth as colab_auth
colab_auth.authenticate_user()
transfer_latest.main()
